# 第6章 久期与凸性 — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch06_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch06_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 6：例6.1/6.4 + 价格—收益率曲线与久期切线


In [ ]:
import numpy as np
from fi.cashflow import make_cashflows
from fi.pricing import price_bond
from fi import risk, plotting
plotting.use_chinese_style()
cfs, ts = make_cashflows(0.03, 3, 1, 100); y = 0.03
P = price_bond(cfs, ts, y, 1); d_mod = risk.modified_duration(cfs, ts, y, 1); conv = risk.convexity(cfs, ts, y, 1)
print(f'P={P:.4f} 修正久期={d_mod:.4f} 凸性={conv:.4f}')
for bp in (-100,-50,50,100):
    dy=bp/10000; tru=(price_bond(cfs,ts,y+dy,1)/P-1)*100; dc=(-d_mod*dy+0.5*conv*dy**2)*100
    print(f'  Δy={bp:+}bp 真实={tru:.3f}% 久期+凸性={dc:.3f}%')
ys = np.linspace(0, 0.06, 121)
fig, ax = plotting.new_axes()
ax.plot(ys*100, [price_bond(cfs,ts,yi,1) for yi in ys], label='真实价格')
ax.plot(ys*100, [P*(1-d_mod*(yi-y)) for yi in ys], '--', label='久期切线')
ax.scatter([y*100],[P], color='k', zorder=5); ax.set_xlabel('收益率 (%)'); ax.set_ylabel('价格'); ax.set_title('真实价格在切线之上(正凸性)'); ax.legend()
fig.tight_layout()


## 编程实验 7：关键利率久期柱状图 + 验证和≈有效久期


In [ ]:
from fi import data
cv = data.load_sample('cgb_yield_curve').set_index('tenor')['yield_pct']/100
key = np.array([2.0,5.0,10.0]); zeros = cv[[2,5,10]].to_numpy()
cf10, t10 = make_cashflows(float(cv[10]), 10, 2, 100)
krd = risk.key_rate_durations(cf10, t10, key, zeros)
for k,v in zip(key, krd): print(f'KRD({int(k)}Y)={v:.4f}')
print('KRD 之和 =', round(krd.sum(),4), ' ≈ 有效久期')
fig, ax = plotting.new_axes(figsize=(7,4))
ax.bar([f'{int(k)}Y' for k in key], krd); ax.set_ylabel('关键利率久期'); ax.set_title('10Y 国债 KRD 分布')
fig.tight_layout()


## 编程实验 8：组合损益归因——平行 vs 曲线情景（用 KRD）


In [ ]:
# 三券组合各 1000 万；情景 A 纯平行+100bp，情景 B 平坦化(短端多升/长端少升)
tenors = [2,5,10]; MV = 1000
dmods = {}
for ten in tenors:
    yi = float(cv[ten]); cf,t = make_cashflows(yi, ten, 2, 100); dmods[ten] = risk.modified_duration(cf,t,yi,2)
dv01 = {ten: dmods[ten]*MV*1e-4 for ten in tenors}
pnl_A = -sum(dv01[ten]*100 for ten in tenors)
# 情景 B：2Y +112.5bp, 5Y +87.5bp, 10Y +62.5bp（整体+75bp、平坦化25bp）
shock = {2:112.5, 5:87.5, 10:62.5}
pnl_B = -sum(dv01[ten]*shock[ten] for ten in tenors)
print(f'情景A 纯平行+100bp 损益 ≈ {pnl_A:.1f} 万元')
print(f'情景B 平坦化(+75bp&陡25bp) 损益 ≈ {pnl_B:.1f} 万元')
print('结论：长端少升的情景B损失更小——只看组合久期会误判，需用 KRD')
print('akshare 真实曲线可替换 cv 重做')
